# Constant Potential in Quantum ESPRESSO
This notebook is based on a tutorial in how to apply a constant potential to a platinum slab in Quantum ESPRESSO.
Dependencies:
- Quantum ESPRESSO v7.4.1
- pw.x, pp.x, average.x
- Environ if you would like to add an implicit solvation model
- ASE any version before 3.25.0
- Numpy any version before 2.0

This notebook is based on the work function exercise created on [compmatphys.org](https://compmatphys.org/wp-content/uploads/2022/09/work-function.pdf).

In [ ]:
# Install libraries
# %pip install "ase<3.25.0"
# %pip install "numpy<2.0"

: 

In [ ]:
# Import libraries
import ase
print(f"ASE version: {ase.__version__}")
import numpy as np
print(f"Numpy version: {np.__version__}")
from ase.io import cif, read
from ase.io.espresso import write_espresso_in
from ase.visualize import view
from ase.build import make_supercell, fcc111_root
from ase import Atoms
import os
import scipy.constants as const
import matplotlib.pyplot as plt

In [ ]:
filename_base = "4-01c"

Once the calculation is done, we can find the Fermi energy of the system.

In [ ]:
fermi_energy = 0
# Search through the output file for the Fermi energy line
with open(f'{filename_base}/{filename_base}.scf.out', 'r') as file:
    for line in file:
        if 'the Fermi energy is' in line:
            fermi_energy = float(line.split()[-2])
print(f"Fermi energy: {fermi_energy} eV")

We can also extract the relaxed structure.

In [ ]:
structures = read(f'{filename_base}/{filename_base}.relax.out', index=':')
relaxed_structure = structures[-1] # stored final relaxed structure for future reference
view(structures, viewer='ngl')

## Prepare and Run pp.x
To calculate and plot the potential across the unit cell, use the pp.x executable on the output of the relax calculation.

In [ ]:
pp_input = f"""&INPUTPP
  outdir = './{filename_base}',
  prefix = '{filename_base}',
  plot_num = 11,
  filplot = 'potential.pot',
/

&PLOT
  iflag = 3,
  output_format = 3,
/
"""

# Save the pp.x input file
with open(f'{filename_base}/{filename_base}.pp.in', 'w') as file:
    file.write(pp_input)

In [ ]:
# Run pp.x
pp_script = f"""#!/bin/bash
#SBATCH -p cpuonly
#SBATCH -n 1
#SBATCH --job-name={filename_base}-pp
#SBATCH -t 5:00:00
#SBATCH -o {filename_base}-pp.out
#SBATCH -e {filename_base}-pp.err

ulimit -s unlimited
ulimit -a  # Print all limits for debugging
export OMP_NUM_THREADS=1

SECONDS=0
module purge
module load psc.allocations.user/1.0
module load intel-oneapi-compilers/2022.1.0 intel-oneapi-mkl/2022.1.0 intel-oneapi-mpi/2021.6.0

PP=/trace/group/dabo/shared/software/qe/qe-7.4.1/build/bin/pp.x

mkdir -p {filename_base}
mpirun -np "${{SLURM_NTASKS}}" $PP -in {filename_base}/{filename_base}.pp.in > {filename_base}/{filename_base}.pp.out 2>&1

duration=$SECONDS
time=`date +%Y%m%d-%H%M%S`
echo "pp is completed in $((duration / 60)) minutes and $((duration % 60)) seconds, date:${{time}}." >> {filename_base}/{filename_base}.pp.out
"""

# Save the job script to a file
with open(f'{filename_base}/{filename_base}-pp.sh', 'w') as file:
    file.write(pp_script)

# Run the job script
!sbatch {filename_base}/{filename_base}-pp.sh

# Plot Potential with average.x
The file specified by `filplot` (i.e. `potential.pot` in this tutorial) contains the potential for every point of the unit cell (including for every point of the vacuum above the slab, which is part of the unit cell too). To calculate the work function, we need the value of the potential in the vacuum a large distance above the surface.

Ideally, we would find a point equally far away from the surface 'below' and 'above', and it would not matter where in that plane we consider the potential. However, this is unlikely to be true in practice, so we use the QE program `average.x` to average the potential over that plane.

In [ ]:
# Input file for average.x
average_input = f"""1
potential.pot
1
800
3
4.4
"""

with open(f'{filename_base}/{filename_base}.average.in', 'w') as file:
    file.write(average_input)

This input file is less intuitive to understand. You can read more about it in the header of the file PP/average.f90, but here is the rough outline of the meaning of each line.

The first and third lines should always be `1` for this application. The first line is the number of files that will be averaged, and the third line is the weight of each file - we are working here with one file only.

The second line is the name for the file that contains the quantity you want to average, i.e. the `potential.pot` file that contains the potential you just calculated.

The fifth line is the plane over which you want to average the potential; the program will then make a large number of slices through the unit cell, always parallel with the chosen plane and the potential will be averaged within each of these slices.

The fourth line is the number of slices. In our example, the unit cell has a length corresponding to 8 bulk layers of platinum. If you take 100 slices between two subsequent layers, then you can follow the average with a good spatial resolution (thus the 8 x 100 = 800 in the input file). It is always wise to test this: a plot of the averages along the unit cell should not change if the slices are taken closer to each other.

The fifth line specifies the orientation of the plane parallel to which you would choose to slice. The definitions are as below:
* `idir = 1`: chosen plane is parallel to that spanned by lattice vectors **b** and **c**
* `idir = 2`: chosen plane is parallel to that spanned by lattice vectors **a** and **c**
* `idir = 3`: chosen plane is parallel to that spanned by lattice vectors **a** and **b**

For our purposes here, we want the plane parallel to the surface of the slab, i.e. `idir = 3`.

The sixth line contains another parameter to eliminate spurious effects. The sequence of averaged potential values will often show small oscillations due to interference effects between the two surfaces that border the vacuum. These are artefacts - for a true surface, there is only vacuum above so there should be no interference from the below surface and there would be no oscillation. Therefore, we will average the potential averages over several slices. A typical size for the number of slices is the distance between two layers. In our example, this is 2.2655 angstroms or 4.28117 Bohr.

In [ ]:
# Run average.x
average_script = f"""#!/bin/bash
#SBATCH -p cpuonly
#SBATCH -n 1
#SBATCH --job-name={filename_base}-average
#SBATCH -t 5:00:00
#SBATCH -o {filename_base}-average.out
#SBATCH -e {filename_base}-average.err

ulimit -s unlimited
ulimit -a  # Print all limits for debugging
export OMP_NUM_THREADS=1

SECONDS=0
module purge
module load psc.allocations.user/1.0
module load intel-oneapi-compilers/2022.1.0 intel-oneapi-mkl/2022.1.0 intel-oneapi-mpi/2021.6.0

AVERAGE=/trace/group/dabo/shared/software/qe/qe-7.4.1/build/bin/average.x

mkdir -p {filename_base}
mpirun -np "${{SLURM_NTASKS}}" $AVERAGE -in {filename_base}/{filename_base}.average.in > {filename_base}/{filename_base}.average.out 2>&1

duration=$SECONDS
time=`date +%Y%m%d-%H%M%S`
echo "average is completed in $((duration / 60)) minutes and $((duration % 60)) seconds, date:${{time}}." >> {filename_base}/{filename_base}.average.out
"""

# Save the job script to a file
with open(f'{filename_base}/{filename_base}-average.sh', 'w') as file:
    file.write(average_script)

# Run the job script
!sbatch {filename_base}/{filename_base}-average.sh

! Note on something that trips me up!
`pp.x` has a known issue with MPI synchronisation/finalisation - the Guassian cube format input/output is done by the master rank, but all ranks must synchronise afterwards, and this can hang indefinitely on some systems or cluster configurations. As a result, run anything that generates cube files on 1 processor.

Besides the pt.average.out file, it will also produce the file avg.dat with three columns:
1. The distance in au along the line through the unit cell
2. The potential averaged over the slce through that point
3. The averages over all slices within the given value (4.28117 a.u.) where the current point is at the centre (i.e. we average over all slices that are 4.28117/2 a.u. before and after the current point). This is a *sliding window* type of averaging.

Now we can plot these two averages along the unit cell.

In [ ]:
# Extract data from avg.dat
with open('avg.dat', 'r') as f:
    data = f.readlines()

# Parse the data
distances = []
one_slice_avg = []
window_avg = []

for line in data:
    values = line.split()
    distances.append(float(values[0]))
    one_slice_avg.append(float(values[1]))
    window_avg.append(float(values[2]))

plt.figure(figsize=(10, 5))
plt.plot(distances, one_slice_avg, label='One Slice Average')
plt.plot(distances, window_avg, label='Window Average')
plt.xlabel('Distance (au)')
plt.ylabel('Potential (Ry)')
plt.legend()

In [ ]:
# Plot again in eV and angstrom

distances_angstrom = [d * const.value('Bohr radius')*(10**10) for d in distances]
one_slice_avg_eV = [v * const.value('Rydberg constant times hc in eV') for v in one_slice_avg]
window_avg_eV = [v * const.value('Rydberg constant times hc in eV') for v in window_avg]

plt.figure(figsize=(10, 5))
plt.plot(distances_angstrom, one_slice_avg_eV, label='One Slice Average')
plt.plot(distances_angstrom, window_avg_eV, label='Window Average')
plt.xlabel('Distance (angstrom)')
plt.ylabel('Potential (eV)')
plt.legend()
plt.show()

Now we can calculate the work function by using the equation:
$$ \Phi = E_{\text{vac}} - E_{\text{F}} $$

In [ ]:
work_function = window_avg_eV[0] - fermi_energy
print(f"Work function: {work_function} eV")

How does this compare to the experimental value for the work function of platinum? (5.12 - 5.93 eV according to [Wikipedia](https://en.wikipedia.org/wiki/Work_function))